# Data Auditing Tutorial

**Before you start:** Run this notebook from the repo (e.g. open it under `Notebooks/tutorials/`). Replace every `path/to/...` and adjust the path template `.../(condition)/(filename).*` to match your folder layout. If you prefer a point-and-click first run, try the [beginner tutorial](01_beginner_interactive.ipynb).

**Why audit?** Microscopy batches often mix image sizes, channel counts, or misaligned masks. Auditing catches that **before** the model runs (saving GPU time and confusing errors). `audit_data()` summarizes height, width, channels, intensity range, read errors, and whether each image matches its mask shape.

See [Best Practices: Data Audit](../../docs/guides/best-practices.md#data-auditing) and [Getting Started: Image Preprocessing](../../docs/getting-started.md#image-preprocessing).

**Author:** Aitor González-Marfil ([@AAitorG](https://github.com/AAitorG))

---

## 1. Setup

In [ ]:
import os
import sys

project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from phenome import PhenoMe, get_metadata_from_path

## 2. Find Files and Audit

The recommended flow: `find_files()` → `audit_data()` → interpret results → `process_images()`.

In [ ]:
pheno = PhenoMe()

# Find files (with optional mask_dir for property presets)
image_dir = "path/to/images"
mask_dir = "path/to/masks"  # Optional; omit if no masks

metadata_fn = get_metadata_from_path(".../(condition)/(filename).*")
file_df = pheno.find_files(
    image_dir,
    mask_dir=mask_dir if os.path.exists(mask_dir) else None,
    metadata_fn=metadata_fn,
    extensions=[".tif", ".tiff", ".png"],
)

print(f"Found {len(file_df)} images")

## 3. Run Audit

`audit_data()` returns a DataFrame with columns: `file_path`, `height`, `width`, `channels`, `min`, `max`, `error`, `shape_match` (image-mask alignment), `mask_height`, `mask_width`, etc.

In [ ]:
audit_df = pheno.audit_data()
print(audit_df.head(10))

## 4. Interpret Results

**What to look for:** Most rows should share one `(height, width, channels)`. Many different shapes → fix acquisition or use `pad_size` / consistent cropping. Non-empty `error` → broken paths or unreadable files. `shape_match` is `False` → image and mask grids do not match (fix filenames or alignment before mask-based features).

| Finding | Typical action |
|---------|----------------|
| One dominant shape | Default `resize_size=224` is often fine |
| Very small images | Consider `pad_size` before resize |
| Multi-channel fluorescence | `channel_mode='split'` |
| RGB / brightfield | `channel_mode='combined'` |

Use the audit to choose:
- **resize_size**: Target size for model input (default 224)
- **pad_size**: Pad small images before resizing
- **channel_mode**: `'split'` for fluorescence, `'combined'` for brightfield/RGB

In [ ]:
if not audit_df.empty:
    # Most common shape
    common = audit_df.groupby(["height", "width", "channels"]).size().idxmax()
    print(f"Most common shape: H={common[0]}, W={common[1]}, C={common[2]}")

    # Check for errors
    errors = audit_df[audit_df["error"].notna()]
    if not errors.empty:
        print(f"Files with errors: {len(errors)}")
        print(errors[["file_path", "error"]].head())

    # Check shape_match when masks exist (use == False so NaN rows without masks are ignored)
    if "shape_match" in audit_df.columns:
        mismatched = audit_df[audit_df["shape_match"] == False]
        if not mismatched.empty:
            print(f"Image-mask shape mismatches: {len(mismatched)}")

## 5. Proceed to Processing

After auditing, run `process_images()` with appropriate parameters. See [Common Workflows](../../docs/examples/workflows.md) for full examples.

In [ ]:
# Example: process with parameters informed by audit
# from phenome import load_dinov2_model
# model, wrapper = load_dinov2_model()
# pheno.process_images(wrapper, resize_size=224, channel_mode="split")

## Next Steps

- [01_data_auditing.ipynb](../examples/01_data_auditing.ipynb): Full data auditing example
- [04_exploratory_analysis.ipynb](../examples/04_exploratory_analysis.ipynb): Downstream analysis after successful auditing and embedding
- [example_advanced_phenotyping.py](../examples/example_advanced_phenotyping.py): Full script-style pipeline
- [Best Practices](../../docs/guides/best-practices.md): Reproducibility and error handling